# MoneyMap — Tutorial Completo

> Conversión de divisas y cálculo de impuestos con **precisión decimal exacta**.

Este notebook cubre **todas** las funcionalidades de MoneyMap, incluyendo casos de borde, funciones avanzadas, integración con pandas/polars y una suite de verificación que replica los tests del paquete.

**Secciones:**
1. Instalación
2. Precisión decimal vs float
3. Conversión de divisas — casos básicos y de borde
4. Registro de tasas custom
5. Cálculo de impuestos
6. Catálogos disponibles
7. Manejo de errores y excepciones
8. Ayuda integrada
9. Integración con pandas
10. Integración con Polars
11. Caso de uso real
12. Suite de verificación (reemplaza el Dockerfile)


## 0 · Instalación

In [24]:
%pip install moneymap[pandas,polars] pytest --quiet

Note: you may need to restart the kernel to use updated packages.


## 1 · Precisión decimal vs float

MoneyMap usa `Decimal` internamente para eliminar los errores de punto flotante que son peligrosos en código financiero.

In [25]:
# ❌ El problema con float
print("float: 0.1 + 0.2 =", 0.1 + 0.2)          # 0.30000000000000004
print("float: == 0.3?",    0.1 + 0.2 == 0.3)     # False  ← peligroso en finanzas

# ✅ Con Decimal
from decimal import Decimal
print()
print("Decimal: 0.1 + 0.2 =", Decimal('0.1') + Decimal('0.2'))                   # 0.3
print("Decimal: == 0.3?",     Decimal('0.1') + Decimal('0.2') == Decimal('0.3')) # True

float: 0.1 + 0.2 = 0.30000000000000004
float: == 0.3? False

Decimal: 0.1 + 0.2 = 0.3
Decimal: == 0.3? True


## 2 · Conversión de divisas

### 2.1 Conversiones básicas

In [26]:
from moneymap import convertir

print(convertir(1000, "MXN", "USD"))   # Decimal('58.65')
print(convertir(100,  "USD", "MXN"))   # Decimal('1705.00')
print(convertir(500,  "EUR", "GBP"))   # Decimal('429.35')
print(convertir(1,    "USD", "JPY"))   # Decimal('149.50')

58.65
1705.00
429.35
149.50


### 2.2 Tipos de entrada aceptados

`convertir` acepta `int`, `float`, `str` y `Decimal` — todos producen resultados `Decimal`.

In [27]:
from decimal import Decimal

print(convertir(100,           "USD", "MXN"))  # int
print(convertir(100.50,        "USD", "MXN"))  # float
print(convertir("100.50",      "USD", "MXN"))  # str
print(convertir(Decimal("100"),"USD", "MXN"))  # Decimal

# Todos son instancias de Decimal
assert all(isinstance(convertir(v, "USD", "MXN"), Decimal) for v in [100, 100.0, "100", Decimal("100")])
print("✅ Todos los tipos de entrada retornan Decimal")

1705.00
1713.53
1713.53
1705.00
✅ Todos los tipos de entrada retornan Decimal


### 2.3 Casos de borde

In [28]:
# Misma divisa → retorna el mismo monto
print("USD→USD:", convertir(100, "USD", "USD"))  # 100.00

# Monto cero
print("Cero:   ", convertir(0, "USD", "MXN"))    # 0.00

# Códigos en minúsculas → funcionan igual
print("Lower:  ", convertir(100, "usd", "mxn"))  # 1705.00

# Verificación de simetría (redondeo puede introducir diferencias mínimas)
ida    = convertir(1000, "MXN", "USD")
vuelta = convertir(float(ida), "USD", "MXN")
print(f"MXN→USD→MXN: 1000 → {ida} → {vuelta}")

USD→USD: 100.00
Cero:    0.00
Lower:   1705.00
MXN→USD→MXN: 1000 → 58.65 → 999.98


### 2.4 Todas las conversiones cruzadas disponibles

In [29]:
from moneymap import divisas_disponibles
import itertools

divisas = divisas_disponibles()
monto   = 100

print(f"{'Origen':<6} {'Destino':<6} {'Resultado':>12}")
print("-" * 28)

# Mostrar algunas conversiones representativas
pares = [("USD","MXN"),("USD","EUR"),("USD","JPY"),("USD","BRL"),
         ("EUR","GBP"),("MXN","BRL"),("ARS","CLP"),("USD","CNY")]
for o, d in pares:
    print(f"{o:<6} {d:<6} {str(convertir(monto, o, d)):>12}")

Origen Destino    Resultado
----------------------------
USD    MXN         1705.00
USD    EUR           92.00
USD    JPY        14950.00
USD    BRL          497.00
EUR    GBP           85.87
MXN    BRL           29.15
ARS    CLP          107.88
USD    CNY          724.00


## 3 · Registro de tasas custom

`registrar_tasa(divisa, referencia, tasa)` — establece cuántas unidades de `divisa` hay por 1 unidad de `referencia`. Sirve para actualizar tasas o agregar divisas nuevas.

In [30]:
from moneymap import registrar_tasa

# ── Actualizar una divisa existente
print("Antes: ", convertir(100, "USD", "MXN"))  # ~1705
registrar_tasa("MXN", "USD", 20.00)              # 1 USD = 20 MXN
print("Después:", convertir(100, "USD", "MXN"))  # 2000.00

# Restaurar
registrar_tasa("MXN", "USD", 17.05)
print("Restaurado:", convertir(100, "USD", "MXN"))

Antes:  1705.00
Después: 2000.00
Restaurado: 1705.00


In [31]:
# ── Agregar una divisa nueva
registrar_tasa("VEF", "USD", 36.50)  # 1 USD = 36.50 VEF
print("VEF en catálogo:", "VEF" in divisas_disponibles())  # True
print("100 USD → VEF:",   convertir(100, "USD", "VEF"))    # 3650.00
print("3650 VEF → USD:",  convertir(3650, "VEF", "USD"))   # 100.00

VEF en catálogo: True
100 USD → VEF: 3650.00
3650 VEF → USD: 100.00


In [32]:
# ── Tasa referenciada a una divisa que no es USD
# Ejemplo: registrar JPY respecto a MXN
registrar_tasa("MXN", "JPY", 0.10)   # 1 JPY = 0.10 MXN
print("1000 JPY → MXN:", convertir(1000, "JPY", "MXN"))  # 100.00

# Restaurar
registrar_tasa("MXN", "USD", 17.05)

1000 JPY → MXN: 100.00


## 4 · Cálculo de impuestos

### 4.1 Funciones principales

In [33]:
from moneymap import impuesto
from moneymap.taxes import total_con_impuesto, tasa_fiscal

monto = 1000

print("tasa_fiscal(Mexico):         ", tasa_fiscal("Mexico"), "%")    # 16
print("impuesto(1000, Mexico):      ", impuesto(monto, "Mexico"))      # 160.00
print("total_con_impuesto(1000,...): ", total_con_impuesto(monto, "Mexico"))  # 1160.00

# Verificar coherencia: impuesto + base = total
assert impuesto(monto, "Mexico") + monto == total_con_impuesto(monto, "Mexico")
print("✅ impuesto + monto == total_con_impuesto")

tasa_fiscal(Mexico):          16 %
impuesto(1000, Mexico):       160.00
total_con_impuesto(1000,...):  1160.00
✅ impuesto + monto == total_con_impuesto


### 4.2 Tabla comparativa — todos los países disponibles

In [34]:
from moneymap import paises_disponibles

monto  = 1000
paises = paises_disponibles()

print(f"{'País':<25} {'Tasa':>6} {'Impuesto':>10} {'Total':>10}")
print("-" * 55)
for pais in paises:
    print(
        f"{pais:<25}"
        f" {str(tasa_fiscal(pais))+'%':>6}"
        f" {str(impuesto(monto, pais)):>10}"
        f" {str(total_con_impuesto(monto, pais)):>10}"
    )

País                        Tasa   Impuesto      Total
-------------------------------------------------------
Alemania                     19%     190.00    1190.00
Argentina                    21%     210.00    1210.00
Australia                    10%     100.00    1100.00
Austria                      20%     200.00    1200.00
Belgica                      21%     210.00    1210.00
Bolivia                      13%     130.00    1130.00
Brasil                       12%     120.00    1120.00
Canada                        5%      50.00    1050.00
Chile                        19%     190.00    1190.00
China                        13%     130.00    1130.00
Colombia                     19%     190.00    1190.00
Corea del Sur                10%     100.00    1100.00
Costa Rica                   13%     130.00    1130.00
Cuba                         10%     100.00    1100.00
Dinamarca                    25%     250.00    1250.00
Ecuador                      15%     150.00    1150.00
El Salvad

### 4.3 Casos de borde en impuestos

In [35]:
# USA — sin IVA federal (0%)
print("USA impuesto:",            impuesto(1000, "USA"))             # 0.00
print("USA total_con_impuesto:",  total_con_impuesto(1000, "USA"))   # 1000.00

# Suiza — tasa decimal (7.7%)
print("Suiza impuesto:",          impuesto(1000, "Suiza"))           # 77.00
print("Suiza tasa:",              tasa_fiscal("Suiza"), "%")         # 7.7

# Monto cero
print("Cero impuesto:",           impuesto(0, "Mexico"))             # 0.00

# Tipos de entrada
print("float input:",             impuesto(999.99, "Mexico"))
print("str input:",               impuesto("500", "España"))

USA impuesto: 0.00
USA total_con_impuesto: 1000.00
Suiza impuesto: 77.00
Suiza tasa: 7.7 %
Cero impuesto: 0.00
float input: 160.00
str input: 105.00


## 5 · Catálogos disponibles

In [36]:
from moneymap import divisas_disponibles, paises_disponibles

d = divisas_disponibles()
p = paises_disponibles()

print(f"Divisas  ({len(d)}): {d}")
print()
print(f"Países   ({len(p)}): {p}")

# Las listas están ordenadas
assert d == sorted(d)
assert p == sorted(p)
print("\n✅ Ambas listas están ordenadas alfabéticamente")

Divisas  (16): ['ARS', 'AUD', 'BRL', 'CAD', 'CHF', 'CLP', 'CNY', 'COP', 'EUR', 'GBP', 'INR', 'JPY', 'MXN', 'PEN', 'USD', 'VEF']

Países   (44): ['Alemania', 'Argentina', 'Australia', 'Austria', 'Belgica', 'Bolivia', 'Brasil', 'Canada', 'Chile', 'China', 'Colombia', 'Corea del Sur', 'Costa Rica', 'Cuba', 'Dinamarca', 'Ecuador', 'El Salvador', 'España', 'Finlandia', 'Francia', 'Grecia', 'Guatemala', 'Honduras', 'India', 'Italia', 'Japon', 'Mexico', 'Nicaragua', 'Noruega', 'Nueva Zelanda', 'Paises Bajos', 'Panama', 'Paraguay', 'Peru', 'Polonia', 'Portugal', 'Reino Unido', 'Republica Dominicana', 'Singapur', 'Suecia', 'Suiza', 'USA', 'Uruguay', 'Venezuela']

✅ Ambas listas están ordenadas alfabéticamente


## 6 · Manejo de errores y excepciones

MoneyMap define su propia jerarquía de excepciones bajo `MoneyMapError`.

In [37]:
from moneymap.exceptions import (
    MoneyMapError,          # clase base
    DivisaNoSoportadaError,
    PaisNoSoportadoError,
    MontoInvalidoError,
    TasaInvalidaError,
)

# Todas las excepciones heredan de MoneyMapError
assert issubclass(DivisaNoSoportadaError, MoneyMapError)
assert issubclass(PaisNoSoportadoError,   MoneyMapError)
assert issubclass(MontoInvalidoError,     MoneyMapError)
assert issubclass(TasaInvalidaError,      MoneyMapError)
print("✅ Jerarquía de excepciones correcta")

✅ Jerarquía de excepciones correcta


In [38]:
# ── Casos de error en convertir
casos_convertir = [
    ("Divisa origen inválida",  lambda: convertir(100, "XYZ", "USD"),     DivisaNoSoportadaError),
    ("Divisa destino inválida", lambda: convertir(100, "USD", "ABC"),     DivisaNoSoportadaError),
    ("Monto negativo",          lambda: convertir(-50, "USD", "MXN"),     MontoInvalidoError),
    ("Monto no numérico",       lambda: convertir("abc", "USD", "MXN"),   MontoInvalidoError),
]

for label, fn, exc_esperada in casos_convertir:
    try:
        fn()
        print(f"❌ {label}: no lanzó excepción")
    except exc_esperada as e:
        print(f"✅ {label}: {type(e).__name__}: {e}")

✅ Divisa origen inválida: DivisaNoSoportadaError: La divisa 'XYZ' no está soportada. Usa divisas_disponibles() para ver las opciones.
✅ Divisa destino inválida: DivisaNoSoportadaError: La divisa 'ABC' no está soportada. Usa divisas_disponibles() para ver las opciones.
✅ Monto negativo: MontoInvalidoError: El monto '-50' no es válido. El monto no puede ser negativo.
✅ Monto no numérico: MontoInvalidoError: El monto 'abc' no es válido. No es un número válido.


In [39]:
# ── Casos de error en impuesto
casos_impuesto = [
    ("País no registrado", lambda: impuesto(100, "Neverland"),    PaisNoSoportadoError),
    ("Monto negativo",     lambda: impuesto(-100, "Mexico"),      MontoInvalidoError),
    ("Monto no numérico",  lambda: impuesto("abc", "Mexico"),     MontoInvalidoError),
]

for label, fn, exc_esperada in casos_impuesto:
    try:
        fn()
        print(f"❌ {label}: no lanzó excepción")
    except exc_esperada as e:
        print(f"✅ {label}: {type(e).__name__}: {e}")

✅ País no registrado: PaisNoSoportadoError: El país 'Neverland' no tiene reglas fiscales registradas. Usa paises_disponibles() para ver las opciones.
✅ Monto negativo: MontoInvalidoError: El monto '-100' no es válido. El monto no puede ser negativo.
✅ Monto no numérico: MontoInvalidoError: El monto 'abc' no es válido. No es un número válido.


In [40]:
# ── Casos de error en registrar_tasa
casos_tasa = [
    ("Tasa negativa",       lambda: registrar_tasa("TST", "USD", -5),          TasaInvalidaError),
    ("Tasa cero",           lambda: registrar_tasa("TST", "USD", 0),            TasaInvalidaError),
    ("Tasa no numérica",    lambda: registrar_tasa("TST", "USD", "no_numero"),  TasaInvalidaError),
    ("Referencia inválida", lambda: registrar_tasa("TST", "XYZ", 10),           DivisaNoSoportadaError),
]

for label, fn, exc_esperada in casos_tasa:
    try:
        fn()
        print(f"❌ {label}: no lanzó excepción")
    except exc_esperada as e:
        print(f"✅ {label}: {type(e).__name__}: {e}")

✅ Tasa negativa: TasaInvalidaError: La tasa de cambio '-5' es inválida. La tasa debe ser mayor que cero.
✅ Tasa cero: TasaInvalidaError: La tasa de cambio '0' es inválida. La tasa debe ser mayor que cero.
✅ Tasa no numérica: TasaInvalidaError: La tasa de cambio 'no_numero' es inválida. No es un número válido.
✅ Referencia inválida: DivisaNoSoportadaError: La divisa 'XYZ' no está soportada. Usa divisas_disponibles() para ver las opciones.


In [41]:
# ── Captura genérica con MoneyMapError (clase base)
casos_genericos = [
    lambda: convertir(100, "????", "USD"),
    lambda: impuesto(100, "Utopía"),
    lambda: registrar_tasa("X", "USD", -1),
]

for fn in casos_genericos:
    try:
        fn()
    except MoneyMapError as e:
        print(f"✅ Capturado por MoneyMapError: {type(e).__name__}")

✅ Capturado por MoneyMapError: DivisaNoSoportadaError
✅ Capturado por MoneyMapError: PaisNoSoportadoError
✅ Capturado por MoneyMapError: TasaInvalidaError


## 7 · Ayuda integrada

`ayuda()` funciona como `man` en bash — índice general o detalle de una función.

In [42]:
from moneymap import ayuda
ayuda()  # Índice de todas las funciones


══════════════════════════════════════════════════════════
  MoneyMap — Referencia de funciones
══════════════════════════════════════════════════════════

  Uso: ayuda('nombre_funcion')  para ver detalles.

  Divisas
    convertir(monto, origen, destino)
      Convierte un monto de una divisa a otra con precisión decimal exacta.
    registrar_tasa(divisa, referencia, tasa)
      Registra o actualiza la tasa de cambio de una divisa.
    divisas_disponibles()
      Devuelve la lista de códigos de divisas soportadas, ordenada alfabéticamente.

  Impuestos
    impuesto(monto, pais)
      Calcula el monto del impuesto (IVA/VAT/GST) según el país.
    total_con_impuesto(monto, pais)
      Calcula el monto total (base + impuesto) para un país.
    tasa_fiscal(pais)
      Devuelve el porcentaje de impuesto registrado para un país.
    paises_disponibles()
      Devuelve la lista de países con reglas fiscales registradas, ordenada alfabéticamente.

  Importar desde el paquete principal:
    f

In [43]:
ayuda("convertir")


──────────────────────────────────────────────────────────
  convertir(monto, origen, destino)
──────────────────────────────────────────────────────────

  Convierte un monto de una divisa a otra con precisión decimal exacta.

  Argumentos
    monto  int | float | str | Decimal
      Valor a convertir. No puede ser negativo.
    origen  str
      Código ISO de la divisa de origen (ej. 'MXN').
    destino  str
      Código ISO de la divisa de destino (ej. 'USD').

  Retorna
    Decimal — resultado redondeado a 2 decimales.

  Lanza
    MontoInvalidoError  —  Si el monto no es numérico o es negativo.
    DivisaNoSoportadaError  —  Si alguna divisa no está registrada.

  Ejemplo
    from moneymap import convertir
    
    convertir(1000, "MXN", "USD")   # → Decimal('58.65')
    convertir(100,  "USD", "MXN")   # → Decimal('1705.00')
    convertir(500,  "EUR", "GBP")   # → Decimal('429.35')

──────────────────────────────────────────────────────────



In [44]:
ayuda("registrar_tasa")


──────────────────────────────────────────────────────────
  registrar_tasa(divisa, referencia, tasa)
──────────────────────────────────────────────────────────

  Registra o actualiza la tasa de cambio de una divisa. Si la divisa no existe la crea; si ya existe la actualiza. La tasa indica cuántas unidades de 'divisa' hay por 1 unidad de 'referencia'.

  Argumentos
    divisa  str
      Código ISO de la divisa a registrar (ej. 'VEF').
    referencia  str
      Código ISO de la divisa base (debe existir, ej. 'USD').
    tasa  int | float | str | Decimal
      Cuántas unidades de 'divisa' equivalen a 1 'referencia'. Debe ser > 0.

  Retorna
    None

  Lanza
    TasaInvalidaError  —  Si la tasa no es un número o es <= 0.
    DivisaNoSoportadaError  —  Si la divisa de referencia no está registrada.

  Ejemplo
    from moneymap import registrar_tasa, convertir
    
    # 1 USD = 17.05 MXN
    registrar_tasa("MXN", "USD", 17.05)
    
    # Agregar divisa nueva: 1 USD = 36.50 VEF
    regis

In [45]:
ayuda("impuesto")


──────────────────────────────────────────────────────────
  impuesto(monto, pais)
──────────────────────────────────────────────────────────

  Calcula el monto del impuesto (IVA/VAT/GST) según el país. Devuelve SOLO el impuesto, no el total.

  Argumentos
    monto  int | float | str | Decimal
      Valor base antes de impuestos.
    pais  str
      Nombre del país (ej. 'Mexico'). Usa paises_disponibles() para ver opciones.

  Retorna
    Decimal — monto del impuesto, redondeado a 2 decimales.

  Lanza
    MontoInvalidoError  —  Si el monto no es numérico o es negativo.
    PaisNoSoportadoError  —  Si el país no tiene reglas fiscales registradas.

  Ejemplo
    from moneymap import impuesto
    
    impuesto(1000, "Mexico")    # → Decimal('160.00')  (IVA 16%)
    impuesto(1000, "España")    # → Decimal('210.00')  (IVA 21%)
    impuesto(1000, "USA")       # → Decimal('0.00')    (sin IVA federal)
    
    total = 1000 + impuesto(1000, "Mexico")  # → 1160

─────────────────────────────

In [46]:
ayuda("total_con_impuesto")


──────────────────────────────────────────────────────────
  total_con_impuesto(monto, pais)
──────────────────────────────────────────────────────────

  Calcula el monto total (base + impuesto) para un país. Equivale a: monto + impuesto(monto, pais).

  Argumentos
    monto  int | float | str | Decimal
      Valor base antes de impuestos.
    pais  str
      Nombre del país.

  Retorna
    Decimal — total con impuesto incluido, redondeado a 2 decimales.

  Lanza
    MontoInvalidoError  —  Si el monto no es válido.
    PaisNoSoportadoError  —  Si el país no está registrado.

  Ejemplo
    from moneymap.taxes import total_con_impuesto
    
    total_con_impuesto(1000, "Mexico")    # → Decimal('1160.00')
    total_con_impuesto(500,  "Alemania")  # → Decimal('595.00')
    total_con_impuesto(1000, "USA")       # → Decimal('1000.00')

──────────────────────────────────────────────────────────



In [47]:
ayuda("tasa_fiscal")


──────────────────────────────────────────────────────────
  tasa_fiscal(pais)
──────────────────────────────────────────────────────────

  Devuelve el porcentaje de impuesto registrado para un país.

  Argumentos
    pais  str
      Nombre del país.

  Retorna
    Decimal — porcentaje (ej. Decimal('16') representa 16%).

  Lanza
    PaisNoSoportadoError  —  Si el país no está registrado.

  Ejemplo
    from moneymap.taxes import tasa_fiscal
    
    tasa_fiscal("Mexico")       # → Decimal('16')
    tasa_fiscal("Reino Unido")  # → Decimal('20')
    tasa_fiscal("Suiza")        # → Decimal('7.7')

──────────────────────────────────────────────────────────



In [48]:
ayuda("divisas_disponibles")


──────────────────────────────────────────────────────────
  divisas_disponibles()
──────────────────────────────────────────────────────────

  Devuelve la lista de códigos de divisas soportadas, ordenada alfabéticamente.

  Retorna
    list[str] — códigos ISO de las divisas registradas.

  Ejemplo
    from moneymap import divisas_disponibles
    
    divisas_disponibles()
    # → ['ARS', 'AUD', 'BRL', 'CAD', 'CHF', 'CLP', 'CNY',
    #     'COP', 'EUR', 'GBP', 'INR', 'JPY', 'MXN', 'PEN', 'USD']

──────────────────────────────────────────────────────────



In [49]:
ayuda("paises_disponibles")


──────────────────────────────────────────────────────────
  paises_disponibles()
──────────────────────────────────────────────────────────

  Devuelve la lista de países con reglas fiscales registradas, ordenada alfabéticamente.

  Retorna
    list[str] — nombres de países disponibles.

  Ejemplo
    from moneymap import paises_disponibles
    
    paises_disponibles()
    # → ['Alemania', 'Argentina', 'Australia', 'Austria',
    #     'Belgica', 'Bolivia', 'Brasil', 'Canada', ...]

──────────────────────────────────────────────────────────



## 8 · Integración con pandas

Se activa con un solo import. Todas las operaciones son **no destructivas** y **encadenables**.

### 8.1 Setup

In [50]:
import pandas as pd
import moneymap.dataframepd   # registra df.moneymap

df = pd.DataFrame({
    "producto": ["Laptop", "Monitor", "Teclado", "Mouse", "Webcam"],
    "precio":   [25000,    8500,      1200,      350,     950],
})
df

,producto,precio
0,Laptop,25000
1,Monitor,8500
2,Teclado,1200
3,Mouse,350
4,Webcam,950


### 8.2 `convertir`

In [51]:
# Nombre de columna por defecto: {col}_{DESTINO}
df.moneymap.convertir(col="precio", origen="MXN", destino="USD")

,producto,precio,precio_USD
0,Laptop,25000,1466.28
1,Monitor,8500,498.53
2,Teclado,1200,70.38
3,Mouse,350,20.53
4,Webcam,950,55.72


In [52]:
# Parámetro `resultado` para nombre personalizado de la columna
df.moneymap.convertir(col="precio", origen="MXN", destino="USD", resultado="precio_dolar")

,producto,precio,precio_dolar
0,Laptop,25000,1466.28
1,Monitor,8500,498.53
2,Teclado,1200,70.38
3,Mouse,350,20.53
4,Webcam,950,55.72


### 8.3 `impuesto` y `total_con_impuesto`

In [53]:
df.moneymap.impuesto(col="precio", pais="Mexico")

,producto,precio,precio_impuesto
0,Laptop,25000,4000.0
1,Monitor,8500,1360.0
2,Teclado,1200,192.0
3,Mouse,350,56.0
4,Webcam,950,152.0


In [54]:
df.moneymap.total_con_impuesto(col="precio", pais="Mexico")

,producto,precio,precio_total
0,Laptop,25000,29000.0
1,Monitor,8500,9860.0
2,Teclado,1200,1392.0
3,Mouse,350,406.0
4,Webcam,950,1102.0


### 8.4 `resumen_fiscal` — impuesto + total + tasa en una sola llamada

In [55]:
df.moneymap.resumen_fiscal(col="precio", pais="España")

,producto,precio,precio_impuesto,precio_total,precio_tasa_pct
0,Laptop,25000,5250.0,30250.0,21.0
1,Monitor,8500,1785.0,10285.0,21.0
2,Teclado,1200,252.0,1452.0,21.0
3,Mouse,350,73.5,423.5,21.0
4,Webcam,950,199.5,1149.5,21.0


### 8.5 Encadenado

In [56]:
(
    df
    .moneymap.convertir(col="precio", origen="MXN", destino="USD")
    .moneymap.impuesto(col="precio_USD", pais="USA")
    .moneymap.total_con_impuesto(col="precio_USD", pais="USA")
)

,producto,precio,precio_USD,precio_USD_impuesto,precio_USD_total
0,Laptop,25000,1466.28,0.0,1466.28
1,Monitor,8500,498.53,0.0,498.53
2,Teclado,1200,70.38,0.0,70.38
3,Mouse,350,20.53,0.0,20.53
4,Webcam,950,55.72,0.0,55.72


In [57]:
# Encadenado con resultado personalizado
(
    df
    .moneymap.convertir(col="precio", origen="MXN", destino="EUR", resultado="precio_eur")
    .moneymap.resumen_fiscal(col="precio_eur", pais="Alemania")
)

,producto,precio,precio_eur,precio_eur_impuesto,precio_eur_total,precio_eur_tasa_pct
0,Laptop,25000,1348.97,256.30,1605.27,19.0
1,Monitor,8500,458.65,87.14,545.79,19.0
2,Teclado,1200,64.75,12.30,77.05,19.0
3,Mouse,350,18.89,3.59,22.48,19.0
4,Webcam,950,51.26,9.74,61.00,19.0


### 8.6 No-destructividad

El DataFrame original nunca se modifica.

In [58]:
columnas_antes = list(df.columns)
_ = df.moneymap.convertir(col="precio", origen="MXN", destino="USD")
_ = df.moneymap.resumen_fiscal(col="precio", pais="Mexico")

assert list(df.columns) == columnas_antes
print(f"✅ Columnas originales intactas: {columnas_antes}")

✅ Columnas originales intactas: ['producto', 'precio']


## 9 · Integración con Polars

Mismo API para `DataFrame` y `LazyFrame`. Las funciones `expr_*` devuelven `pl.Expr`.

### 9.1 Setup

In [59]:
import polars as pl
import moneymap.dataframepl as mm

df_pl = pl.DataFrame({
    "producto": ["Laptop", "Monitor", "Teclado", "Mouse", "Webcam"],
    "precio":   [25000,    8500,      1200,      350,     950],
})
df_pl

producto,precio
str,i64
"""Laptop""",25000
"""Monitor""",8500
"""Teclado""",1200
"""Mouse""",350
"""Webcam""",950


### 9.2 `convertir`, `impuesto`, `total_con_impuesto`, `resumen_fiscal`

In [60]:
mm.convertir(df_pl, col="precio", origen="MXN", destino="USD")

producto,precio,precio_USD
str,i64,f64
"""Laptop""",25000,1466.28
"""Monitor""",8500,498.53
"""Teclado""",1200,70.38
"""Mouse""",350,20.53
"""Webcam""",950,55.72


In [61]:
mm.impuesto(df_pl, col="precio", pais="Mexico")

producto,precio,precio_impuesto
str,i64,f64
"""Laptop""",25000,4000.0
"""Monitor""",8500,1360.0
"""Teclado""",1200,192.0
"""Mouse""",350,56.0
"""Webcam""",950,152.0


In [62]:
mm.total_con_impuesto(df_pl, col="precio", pais="Mexico")

producto,precio,precio_total
str,i64,f64
"""Laptop""",25000,29000.0
"""Monitor""",8500,9860.0
"""Teclado""",1200,1392.0
"""Mouse""",350,406.0
"""Webcam""",950,1102.0


In [63]:
mm.resumen_fiscal(df_pl, col="precio", pais="España")

producto,precio,precio_impuesto,precio_total,precio_tasa_pct
str,i64,f64,f64,f64
"""Laptop""",25000,5250.0,30250.0,21.0
"""Monitor""",8500,1785.0,10285.0,21.0
"""Teclado""",1200,252.0,1452.0,21.0
"""Mouse""",350,73.5,423.5,21.0
"""Webcam""",950,199.5,1149.5,21.0


### 9.3 LazyFrame — evaluación diferida

In [64]:
# Todo se ejecuta al llamar .collect()
resultado = (
    df_pl.lazy()
    .pipe(mm.convertir, col="precio", origen="MXN", destino="USD")
    .pipe(mm.impuesto,  col="precio_USD", pais="USA")
    .pipe(mm.total_con_impuesto, col="precio_USD", pais="USA")
    .collect()
)
resultado

c:\Users\LARIG\anaconda3\Lib\site-packages\moneymap\dataframepl.py:66: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  cols = df.columns


producto,precio,precio_USD,precio_USD_impuesto,precio_USD_total
str,i64,f64,f64,f64
"""Laptop""",25000,1466.28,0.0,1466.28
"""Monitor""",8500,498.53,0.0,498.53
"""Teclado""",1200,70.38,0.0,70.38
"""Mouse""",350,20.53,0.0,20.53
"""Webcam""",950,55.72,0.0,55.72


In [65]:
# LazyFrame retorna LazyFrame hasta .collect()
lazy = df_pl.lazy().pipe(mm.convertir, col="precio", origen="MXN", destino="USD")
print("Tipo antes de collect:", type(lazy).__name__)    # LazyFrame
print("Tipo después de collect:", type(lazy.collect()).__name__)  # DataFrame

Tipo antes de collect: LazyFrame
Tipo después de collect: DataFrame


### 9.4 Expresiones (`expr_*`) — múltiples columnas en una sola pasada

In [66]:
df_pl.with_columns(
    mm.expr_convertir("precio", "MXN", "USD").alias("precio_USD"),
    mm.expr_convertir("precio", "MXN", "EUR").alias("precio_EUR"),
    mm.expr_impuesto("precio", "Mexico").alias("iva"),
    mm.expr_total_con_impuesto("precio", "Mexico").alias("precio_total"),
)

producto,precio,precio_USD,precio_EUR,iva,precio_total
str,i64,f64,f64,f64,f64
"""Laptop""",25000,1466.28,1348.97,4000.0,29000.0
"""Monitor""",8500,498.53,458.65,1360.0,9860.0
"""Teclado""",1200,70.38,64.75,192.0,1392.0
"""Mouse""",350,20.53,18.89,56.0,406.0
"""Webcam""",950,55.72,51.26,152.0,1102.0


## 10 · Caso de uso real

Tabla de ventas internacionales: convertir a USD y calcular el impuesto local de cada país.

In [67]:
ventas = pd.DataFrame({
    "orden":    [1001,      1002,     1003,     1004,          1005],
    "producto": ["Laptop",  "Curso",  "Plugin", "Libro",       "Suscripción"],
    "monto":    [1200,      49,       19,        25,            9.99],
    "divisa":   ["USD",     "EUR",    "USD",     "GBP",         "USD"],
    "pais":     ["Mexico",  "España", "USA",     "Reino Unido", "Argentina"],
})

# Convertir cada monto a USD según su divisa de origen
ventas["monto_usd"] = ventas.apply(
    lambda r: float(convertir(r["monto"], r["divisa"], "USD")), axis=1
)

# Impuesto local de cada país
ventas["impuesto_usd"] = ventas.apply(
    lambda r: float(impuesto(r["monto_usd"], r["pais"])), axis=1
)

# Total con impuesto
ventas["total_usd"] = ventas.apply(
    lambda r: float(total_con_impuesto(r["monto_usd"], r["pais"])), axis=1
)

ventas

,orden,producto,monto,divisa,pais,monto_usd,impuesto_usd,total_usd
0,1001,Laptop,1200.00,USD,Mexico,1200.00,192.00,1392.00
1,1002,Curso,49.00,EUR,España,53.26,11.18,64.44
2,1003,Plugin,19.00,USD,USA,19.00,0.00,19.00
3,1004,Libro,25.00,GBP,Reino Unido,31.65,6.33,37.98
4,1005,Suscripción,9.99,USD,Argentina,9.99,2.10,12.09


In [68]:
print(f"Subtotal (USD):        ${ventas['monto_usd'].sum():.2f}")
print(f"Total impuestos (USD): ${ventas['impuesto_usd'].sum():.2f}")
print(f"Total con impuestos:   ${ventas['total_usd'].sum():.2f}")

Subtotal (USD):        $1313.90
Total impuestos (USD): $211.61
Total con impuestos:   $1525.51


## 11 · Suite de verificación (equivalente al Dockerfile)

Esta sección replica exactamente lo que hace `docker run --rm moneymap pytest` — ejecuta los 41 tests oficiales del paquete directamente desde el notebook, sin necesidad de Docker.

> **Nota sobre Docker:** el `Dockerfile` del repo construye la imagen, instala las dependencias y corre `pytest` sobre `moneymap/test_moneymap.py`. La celda siguiente hace exactamente lo mismo en el entorno actual.

In [69]:
import subprocess, sys

# Localizar el archivo de tests del paquete instalado
import importlib.util, pathlib
spec      = importlib.util.find_spec("moneymap")
pkg_dir   = pathlib.Path(spec.origin).parent
test_file = pkg_dir / "test_moneymap.py"

print(f"Ejecutando: {test_file}\n")

result = subprocess.run(
    [sys.executable, "-m", "pytest", str(test_file), "-v", "--tb=short", "--no-header"],
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode == 0:
    print("\n✅ Todos los tests pasaron — MoneyMap funciona correctamente")
else:
    print("\n❌ Algunos tests fallaron")
    print(result.stderr)

Ejecutando: c:\Users\LARIG\anaconda3\Lib\site-packages\moneymap\test_moneymap.py

============================= test session starts =============================
collecting ... collected 41 items

..\..\..\..\..\..\..\anaconda3\Lib\site-packages\moneymap\test_moneymap.py::TestConvertir::test_misma_divisa_retorna_mismo_monto PASSED [  2%]
..\..\..\..\..\..\..\anaconda3\Lib\site-packages\moneymap\test_moneymap.py::TestConvertir::test_conversion_mxn_a_usd PASSED [  4%]
..\..\..\..\..\..\..\anaconda3\Lib\site-packages\moneymap\test_moneymap.py::TestConvertir::test_conversion_usd_a_mxn PASSED [  7%]
..\..\..\..\..\..\..\anaconda3\Lib\site-packages\moneymap\test_moneymap.py::TestConvertir::test_conversion_acepta_int PASSED [  9%]
..\..\..\..\..\..\..\anaconda3\Lib\site-packages\moneymap\test_moneymap.py::TestConvertir::test_conversion_acepta_float PASSED [ 12%]
..\..\..\..\..\..\..\anaconda3\Lib\site-packages\moneymap\test_moneymap.py::TestConvertir::test_conversion_acepta_string PASSED [ 14

### 11.1 Verificación manual — resumen de funciones cubiertas

Checklist completo de todas las funciones públicas de MoneyMap:

In [70]:
from decimal import Decimal
from moneymap import (
    convertir, registrar_tasa, divisas_disponibles,
    impuesto, paises_disponibles, ayuda,
)
from moneymap.taxes import total_con_impuesto, tasa_fiscal
from moneymap.exceptions import (
    MoneyMapError, DivisaNoSoportadaError, PaisNoSoportadoError,
    MontoInvalidoError, TasaInvalidaError,
)
import moneymap.dataframepd
import moneymap.dataframepl as mm
import pandas as pd, polars as pl

resultados = []

def check(nombre, condicion):
    estado = "✅" if condicion else "❌"
    resultados.append(condicion)
    print(f"{estado} {nombre}")

# ── convertir
check("convertir: MXN→USD",         convertir(1000, "MXN", "USD") == Decimal("58.65"))
check("convertir: USD→MXN",         convertir(100,  "USD", "MXN") == Decimal("1705.00"))
check("convertir: misma divisa",     convertir(100, "USD", "USD") == Decimal("100.00"))
check("convertir: monto cero",       convertir(0, "USD", "MXN") == Decimal("0.00"))
check("convertir: minúsculas OK",    convertir(100, "usd", "mxn") == convertir(100, "USD", "MXN"))
check("convertir: retorna Decimal",  isinstance(convertir(100, "USD", "EUR"), Decimal))
check("convertir: acepta float",     isinstance(convertir(1.5, "USD", "MXN"), Decimal))
check("convertir: acepta str",       isinstance(convertir("100", "USD", "MXN"), Decimal))
check("convertir: acepta Decimal",   isinstance(convertir(Decimal("50"), "USD", "MXN"), Decimal))

# ── registrar_tasa
registrar_tasa("TST", "USD", 99.0)
check("registrar_tasa: nueva divisa",   "TST" in divisas_disponibles())
check("registrar_tasa: conversión OK",  convertir(1, "USD", "TST") == Decimal("99.00"))
registrar_tasa("MXN", "USD", 17.05)  # restaurar

# ── impuesto
check("impuesto: Mexico 16%",    impuesto(1000, "Mexico") == Decimal("160.00"))
check("impuesto: USA 0%",        impuesto(1000, "USA") == Decimal("0.00"))
check("impuesto: España 21%",    impuesto(1000, "España") == Decimal("210.00"))
check("impuesto: monto cero",    impuesto(0, "Mexico") == Decimal("0.00"))

# ── total_con_impuesto
check("total_con_impuesto: Mexico",  total_con_impuesto(1000, "Mexico") == Decimal("1160.00"))
check("total_con_impuesto: USA",     total_con_impuesto(1000, "USA") == Decimal("1000.00"))
check("total_con_impuesto: coherencia",
      total_con_impuesto(500, "España") == Decimal("500") + impuesto(500, "España"))

# ── tasa_fiscal
check("tasa_fiscal: Mexico",  tasa_fiscal("Mexico") == Decimal("16"))
check("tasa_fiscal: Suecia",  tasa_fiscal("Suecia") == Decimal("25"))
check("tasa_fiscal: Suiza",   tasa_fiscal("Suiza")  == Decimal("7.7"))

# ── catálogos
check("divisas_disponibles: es lista ordenada", divisas_disponibles() == sorted(divisas_disponibles()))
check("paises_disponibles: es lista ordenada",  paises_disponibles() == sorted(paises_disponibles()))
check("divisas_disponibles: contiene USD/MXN",  all(d in divisas_disponibles() for d in ["USD","MXN","EUR"]))
check("paises_disponibles: +35 países",         len(paises_disponibles()) >= 35)

# ── excepciones
def raises(fn, exc):
    try: fn(); return False
    except exc: return True
    except: return False

check("exc: DivisaNoSoportadaError",  raises(lambda: convertir(1, "XYZ", "USD"), DivisaNoSoportadaError))
check("exc: PaisNoSoportadoError",    raises(lambda: impuesto(1, "Neverland"),    PaisNoSoportadoError))
check("exc: MontoInvalidoError neg",  raises(lambda: convertir(-1, "USD", "MXN"), MontoInvalidoError))
check("exc: MontoInvalidoError str",  raises(lambda: convertir("x", "USD", "MXN"),MontoInvalidoError))
check("exc: TasaInvalidaError neg",   raises(lambda: registrar_tasa("X","USD",-1),TasaInvalidaError))
check("exc: TasaInvalidaError cero",  raises(lambda: registrar_tasa("X","USD", 0),TasaInvalidaError))
check("exc: base MoneyMapError",      raises(lambda: convertir(1,"???","USD"),    MoneyMapError))

# ── pandas
df_test = pd.DataFrame({"p": [1000, 2000]})
r_pd = df_test.moneymap.convertir(col="p", origen="MXN", destino="USD")
check("pandas: convertir agrega columna",    "p_USD" in r_pd.columns)
r_pd2 = df_test.moneymap.convertir(col="p", origen="MXN", destino="USD", resultado="dolar")
check("pandas: param resultado funciona",    "dolar" in r_pd2.columns)
r_pd3 = df_test.moneymap.resumen_fiscal(col="p", pais="Mexico")
check("pandas: resumen_fiscal cols OK",
      all(c in r_pd3.columns for c in ["p_impuesto","p_total","p_tasa_pct"]))
check("pandas: no-destructivo",              list(df_test.columns) == ["p"])

# ── polars
df_pl_test = pl.DataFrame({"p": [1000, 2000]})
r_pl = mm.convertir(df_pl_test, col="p", origen="MXN", destino="USD")
check("polars: convertir agrega columna",    "p_USD" in r_pl.columns)
r_lazy = df_pl_test.lazy().pipe(mm.convertir, col="p", origen="MXN", destino="USD")
check("polars: LazyFrame → LazyFrame",       isinstance(r_lazy, pl.LazyFrame))
r_expr = df_pl_test.with_columns(
    mm.expr_convertir("p","MXN","USD").alias("u"),
    mm.expr_impuesto("p","Mexico").alias("i"),
    mm.expr_total_con_impuesto("p","Mexico").alias("t"),
)
check("polars: expr_* en with_columns",      all(c in r_expr.columns for c in ["u","i","t"]))

# ── resumen
total   = len(resultados)
pasados = sum(resultados)
print(f"\n{'='*50}")
print(f"Resultado: {pasados}/{total} verificaciones pasaron")
if pasados == total:
    print("✅ MoneyMap funciona correctamente en todas sus funciones")
else:
    print(f"❌ {total - pasados} verificaciones fallaron")

✅ convertir: MXN→USD
✅ convertir: USD→MXN
✅ convertir: misma divisa
✅ convertir: monto cero
✅ convertir: minúsculas OK
✅ convertir: retorna Decimal
✅ convertir: acepta float
✅ convertir: acepta str
✅ convertir: acepta Decimal
✅ registrar_tasa: nueva divisa
✅ registrar_tasa: conversión OK
✅ impuesto: Mexico 16%
✅ impuesto: USA 0%
✅ impuesto: España 21%
✅ impuesto: monto cero
✅ total_con_impuesto: Mexico
✅ total_con_impuesto: USA
✅ total_con_impuesto: coherencia
✅ tasa_fiscal: Mexico
✅ tasa_fiscal: Suecia
✅ tasa_fiscal: Suiza
✅ divisas_disponibles: es lista ordenada
✅ paises_disponibles: es lista ordenada
✅ divisas_disponibles: contiene USD/MXN
✅ paises_disponibles: +35 países
✅ exc: DivisaNoSoportadaError
✅ exc: PaisNoSoportadoError
✅ exc: MontoInvalidoError neg
✅ exc: MontoInvalidoError str
✅ exc: TasaInvalidaError neg
✅ exc: TasaInvalidaError cero
✅ exc: base MoneyMapError
✅ pandas: convertir agrega columna
✅ pandas: param resultado funciona
✅ pandas: resumen_fiscal cols OK
✅ pandas: 

c:\Users\LARIG\anaconda3\Lib\site-packages\moneymap\dataframepl.py:66: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  cols = df.columns
